# H02 — Linear systems: the oscillator a detector is made of

> **References**
>
> (Saulson 2017), (Oppenheim et al. 1997), (Allen and others 2000)

## What you do here

A detector is built from oscillators: every mirror hangs from a pendulum and every fibre rings. This laboratory asks five questions about one oscillator, the damped mirror on a spring. How do we find its transfer function? How do we use it for any input? What is a resonance? What does it mean to work at the resonance and away from it? What changes when we hang several pendulums one under the other?

All the code is given. In every task you change the physical parameters on the line marked `# change these`, run the cells again and answer one question on the physics.

The oscillator is a mirror of mass $m$ hanging from a spring of constant $k$, with a friction of coefficient $b$. The ground moves the point of suspension by $x_{\mathrm{in}}(t)$, the input. The mirror moves by $x_{\mathrm{out}}(t)$, the output. We first let the free oscillator ring, only as far as we need its natural frequency, its quality factor and its ring-down time. Then the ground moves.

> **The symbols**
>
> - **$m$** — the mass of the mirror, in kg
> - **$k$** — the spring constant, in N/m
> - **$b$** — the damping coefficient, in kg/s: the friction on the mirror is $-b\,\dot{x}_{\mathrm{out}}$
> - **$x_{\mathrm{in}}(t)$** — the input: the motion of the ground, which the suspension point follows
> - **$x_{\mathrm{out}}(t)$** — the output: the motion of the mirror

| Part | The question | Tasks |
|---|---|---|
| A · The transfer function | How do we find $G(\omega)$ from the equation and how do we check it? | 1 |
| B · Any input | How does the mirror answer any ground motion? | 2 |
| C · The resonance | What is a resonance: its height, its phase, its width? | 3 and 4 |
| D · At the resonance and away from it | What do we gain at the resonance and what do we pay? How does the mirror isolate far above it? | 5, 6 and 7 |
| E · Several pendulums | How many stages and how long must they be? | 8 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

> **How to use this notebook**
>
> Run the cells in order. The first code cell of a task defines its functions; the next one calls them and draws the result. Change one parameter at a time on the line `# change these`, run the cell again and look at what moves. Answer the question of the task before you open the solution. The line **Needs** says which functions of earlier tasks a task uses.

## The free oscillator

The ground is at rest, $x_{\mathrm{in}} = 0$. We pull the mirror to $x_{\mathrm{out}} = 1$ and let it go. The spring pulls with $-k\,x_{\mathrm{out}}$ and the friction with $-b\,\dot{x}_{\mathrm{out}}$. Newton's second law gives

$$
m\,\ddot{x}_{\mathrm{out}} + b\,\dot{x}_{\mathrm{out}} + k\,x_{\mathrm{out}} = 0 .
$$

We divide by $m$. So $\omega_{0}^{2} = k/m$ and $b/m = \omega_{0}/Q$:

$$
\ddot{x}_{\mathrm{out}} + \frac{\omega_{0}}{Q}\,\dot{x}_{\mathrm{out}} + \omega_{0}^{2}\,x_{\mathrm{out}} = 0 .
$$

Lecture 8 solves it. The mirror rings and fades:

$$
x_{\mathrm{out}}(t) = A\,\mathrm{e}^{-t/\tau_{0}}\cos\left(\omega_{d}t + \varphi\right),
\qquad
\tau_{0} = \frac{2m}{b} = \frac{2Q}{\omega_{0}} .
$$

The function `oscillator_parameters` returns $f_{0}$, $Q$ and $\tau_{0}$ from $m$, $k$ and $b$. The function `rhs` is Newton's second law written for a solver: the state is the position $x_{\mathrm{out}}$ and the velocity $v = \dot{x}_{\mathrm{out}}$, with $\dot{v} = \left[-k(x_{\mathrm{out}} - x_{\mathrm{in}}) - b\,v\right]/m$. Every later task integrates it with `solve_ivp`. The cell below lets the mirror go and draws its motion with the envelope $\pm\mathrm{e}^{-t/\tau_{0}}$. Nothing is asked here. The three numbers it prints are the ones the tasks use.

> **The symbols**
>
> - **$\omega_{0} = \sqrt{k/m}$** — the natural frequency, in radians per second; in hertz $f_{0} = \omega_{0}/2\pi$
> - **$Q = m\omega_{0}/b = \sqrt{km}/b$** — the quality factor: the larger $Q$, the weaker the friction
> - **$\tau_{0}$** — the ring-down time: the amplitude falls by a factor $\mathrm{e}$ in $\tau_{0}$
> - **$\omega_{d}$** — the frequency of the ringing, $\omega_{0}\sqrt{1 - 1/4Q^{2}}$: for $Q \gg 1$ it is $\omega_{0}$
> - **$A$, $\varphi$** — the amplitude and the phase, fixed by the start
> - **$v$** — the velocity of the mirror, $\dot{x}_{\mathrm{out}}$

In [ ]:
def oscillator_parameters(m, k, b):
    """Natural frequency f0 (Hz), quality factor Q and decay time tau0 (s)."""
    w0 = np.sqrt(k / m)
    q = np.sqrt(k * m) / b if b > 0 else np.inf          # b = 0: no friction
    tau0 = 2 * m / b if b > 0 else np.inf
    params = (w0 / (2 * np.pi), q, tau0)
    return params


def rhs(t, state, m, k, b, x_in):
    """Newton's second law for the mirror: state = (x_out, v), x_in(t) the ground."""
    x_out, v = state
    a = (-k * (x_out - x_in(t)) - b * v) / m
    derivatives = [v, a]
    return derivatives


def ground_at_rest(t):
    x_in = 0.0
    return x_in

In [ ]:
# the oscillator of the laboratory: mass m (kg), spring constant k (N/m), damping b (kg/s)
m, k, b = 1.0, 39.5, 0.21

f0, Q, tau0 = oscillator_parameters(m, k, b)
print(f"f0 = {f0:.5f} Hz   Q = {Q:.2f}   tau0 = {tau0:.3f} s   swings in one decay time = {tau0 * f0:.2f}")

t = np.linspace(0, 3 * tau0, 6001)
sol = solve_ivp(rhs, (0, t[-1]), [1.0, 0.0], t_eval=t, args=(m, k, b, ground_at_rest),
                rtol=1e-10, atol=1e-12)
x_out = sol.y[0]

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(t, x_out, color=P['BLUE'], lw=1.8)
ax.plot(t, np.exp(-t / tau0), color=P['BRICK'], lw=1.4, label=r'$\pm\,e^{-t/\tau_0}$')
ax.plot(t, -np.exp(-t / tau0), color=P['BRICK'], lw=1.4)
ax.axvline(tau0, color=P['GREY_MID'], lw=0.8, ls=':')
ax.set(xlabel='time (s)', ylabel=r'$x_{\mathrm{out}}$ (arbitrary units)')
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), frameon=False)
plt.show()

## Part A · How we find the transfer function

The ground moves and the mirror answers. How much of the ground motion reaches the mirror, at each frequency? The transfer function says it. We find it from the equation and then we check it against the motion itself.

### Task 1 · The transfer function from the equation, checked against the motion

Now the ground moves the support by $x_{\mathrm{in}}(t)$. The spring connects the mirror to the support, so its force depends on the stretch, $-k(x_{\mathrm{out}} - x_{\mathrm{in}})$. The friction acts on the motion of the mirror, $-b\,\dot{x}_{\mathrm{out}}$. Newton's second law adds them:

$$
m\,\ddot{x}_{\mathrm{out}} = -k\left(x_{\mathrm{out}} - x_{\mathrm{in}}\right) - b\,\dot{x}_{\mathrm{out}} .
$$

We move the terms in $x_{\mathrm{out}}$ to the left and divide by $m$:

$$
\ddot{x}_{\mathrm{out}} + \frac{\omega_{0}}{Q}\,\dot{x}_{\mathrm{out}} + \omega_{0}^{2}\,x_{\mathrm{out}} = \omega_{0}^{2}\,x_{\mathrm{in}}(t) .
$$

The term $\omega_{0}^{2}\,x_{\mathrm{in}}$ is the force of the moving support per unit mass. With $x_{\mathrm{in}} = 0$ it is the free oscillator.

How do we find $G$? We take the Fourier transform of the equation. Each time derivative becomes a factor $\mathrm{i}\omega$, so $\ddot{x}_{\mathrm{out}} \to -\omega^{2}X_{\mathrm{out}}$:

$$
\left(\omega_{0}^{2} - \omega^{2} + \mathrm{i}\,\omega\omega_{0}/Q\right) X_{\mathrm{out}}(\omega) = \omega_{0}^{2}\,X_{\mathrm{in}}(\omega) .
$$

We divide by $X_{\mathrm{in}}$:

$$
G(\omega) = \frac{X_{\mathrm{out}}}{X_{\mathrm{in}}}
= \frac{\omega_{0}^{2}}{\omega_{0}^{2} - \omega^{2} + \mathrm{i}\,\omega\omega_{0}/Q} .
$$

Is it true? We move the ground as $x_{\mathrm{in}} = \cos 2\pi f t$. After the transient has died, in a few decay times, the mirror moves as $|G|\cos(2\pi f t + \phi)$, with $|G|$ the modulus and $\phi$ the phase of $G$. The function `transfer` computes the formula. The function `steady_response` does not use it. It integrates Newton's second law for twelve decay times, so that the transient has died. Then it reads twenty cycles of the mirror and projects them on a cosine and a sine. This gives the amplitude and the phase. It returns also the mean power the drive gives to the mirror, which task 3 uses. The cell sweeps $f$ from $f_{0}/10$ to $10f_{0}$ and draws the integration as dots on the formula. It takes about half a minute at $Q = 30$.

**Needs.** `oscillator_parameters` and `rhs` from the free oscillator.

> **The symbols**
>
> - **$f$** — the frequency of the ground motion, in hertz; $\omega = 2\pi f$
> - **$X_{\mathrm{in}}(\omega)$, $X_{\mathrm{out}}(\omega)$** — the Fourier transforms of the input and of the output
> - **$G(\omega)$** — the transfer function: $|G|$ is the amplitude of the mirror for a ground amplitude of 1 and $\phi$ its phase
> - **$\bar{P}$** — the mean power the drive gives to the mirror, in watts: the force the spring passes, $k\,x_{\mathrm{in}}$, times the velocity $v$

In [ ]:
def transfer(f, m, k, b):
    """G = X_out / X_in of the oscillator at the frequencies f (Hz)."""
    w = 2 * np.pi * f
    g = k / (k - m * w ** 2 + 1j * b * w)
    return g


def steady_response(m, k, b, f, samples=40, cycles=20):
    """Amplitude, phase (deg) and mean power of the mirror for x_in = cos 2 pi f t."""
    f0, q, tau0 = oscillator_parameters(m, k, b)
    w = 2 * np.pi * f
    start = np.ceil(12 * tau0 * f) / f                   # twelve decay times
    t = start + np.arange(cycles * samples) / (samples * f)

    def x_in(s):
        return np.cos(w * s)

    sol = solve_ivp(rhs, (0, t[-1]), [0.0, 0.0], t_eval=t, args=(m, k, b, x_in),
                    rtol=1e-8, atol=1e-11)
    x_out, v = sol.y
    c, d = 2 * np.mean(x_out * np.cos(w * t)), 2 * np.mean(x_out * np.sin(w * t))
    response = (np.hypot(c, d), np.degrees(np.arctan2(-d, c)), np.mean(k * x_in(t) * v))
    return response

In [ ]:
# change these: mass m (kg), spring constant k (N/m), damping b (kg/s)
m, k, b = 1.0, 39.5, 0.21

f0, Q, tau0 = oscillator_parameters(m, k, b)
f_sweep = f0 * np.unique(np.round(np.concatenate([np.geomspace(0.1, 10, 13),
                                                  [0.9, 0.95, 0.98, 1.02, 1.05, 1.1]]), 6))
amplitude, phase, power = np.array([steady_response(m, k, b, fi) for fi in f_sweep]).T

f_fine = f0 * np.geomspace(0.1, 10, 2000)
G = transfer(f_fine, m, k, b)
G_sweep = transfer(f_sweep, m, k, b)
print(f"Q = {Q:.2f}   f0 = {f0:.5f} Hz")
print(f"amplitude, integration against |G|: largest relative difference "
      f"{np.max(np.abs(amplitude / np.abs(G_sweep) - 1)):.1e}")
print(f"phase, integration against the phase of G: largest difference "
      f"{np.max(np.abs(phase - np.angle(G_sweep, deg=True))):.1e} degrees")

fig, (axa, axb) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axa.loglog(f_fine, np.abs(G), color=P['BLUE'], lw=1.5, label='$|G|$, formula')
axa.loglog(f_sweep, amplitude, 'o', color=P['INK'], ms=3.5, label='integration')
axa.set(ylabel=r'amplitude of $x_{\mathrm{out}}$')
axa.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
axb.semilogx(f_fine, np.angle(G, deg=True), color=P['BLUE'], lw=1.5)
axb.semilogx(f_sweep, phase, 'o', color=P['INK'], ms=3.5)
axb.set(xlabel='frequency of the ground motion (Hz)', ylabel=r'phase $\phi$ (degrees)',
        yticks=[-180, -135, -90, -45, 0])
for ax in (axa, axb):
    ax.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
plt.show()

**What to look for.** The dots of the integration lie on the formula: $G$, found from the equation with one division, is what the mirror does. Three regimes. Well below $f_{0}$ the amplitude is 1 and the phase $0^{\circ}$: the mirror follows the ground. Near $f_{0}$ the amplitude grows and the phase passes $-90^{\circ}$. Well above $f_{0}$ the amplitude falls and the phase goes to $-180^{\circ}$.

**Question.** Before you run the cell, read from the formula $|G|$ and $\phi$ at $\omega \ll \omega_{0}$, at $\omega = \omega_{0}$ and at $\omega \gg \omega_{0}$. Then change $k$, then $m$, then $b$. Does the formula still describe the integration? Which two numbers fix the whole curve?

<details>
<summary>Solution</summary>

For $\omega \ll \omega_{0}$ the denominator is $\omega_{0}^{2}$ and $G = 1$: $|G| = 1$ and $\phi = 0^{\circ}$. At $\omega = \omega_{0}$ the real part of the denominator is zero and $G = \omega_{0}^{2}/(\mathrm{i}\,\omega_{0}^{2}/Q) = -\mathrm{i} Q$: $|G| = Q$ and $\phi = -90^{\circ}$. For $\omega \gg \omega_{0}$ the denominator is $-\omega^{2}$ and $G = -\omega_{0}^{2}/\omega^{2}$: $|G| = (\omega_{0}/\omega)^{2}$ and $\phi = -180^{\circ}$.

The integration agrees with the formula for every choice: the amplitude to about a few parts in $10^{6}$ and the phase to a few hundredths of a degree, the tolerance of the solver and of the projection. The equation is linear with constant coefficients, so the answer to a cosine is a cosine of the same frequency and $G$ is all we need to know.

The curve depends on two numbers only. The frequency axis is $f/f_{0}$ and the shape is fixed by $Q$. A stiffer spring or a lighter mass moves the whole curve along the frequency axis without changing its shape at fixed $Q$. Less friction raises the peak and does not move it.

</details>


## Part B · Any input

One transfer function, found once, answers every input. The next task uses it on two sinusoids and on a random ground motion.

### Task 2 · Any input: $X_{\mathrm{out}} = G\,X_{\mathrm{in}}$

An input that is not one cosine is a sum of cosines. The Fourier transform finds the sum. If the system is linear the output for a sum of inputs is the sum of the outputs. So in frequency the output is one product, $X_{\mathrm{out}} = G\,X_{\mathrm{in}}$, for any input.

The function `respond` integrates the mirror from rest for any ground motion. The cell first moves the ground with two sinusoids, $\cos(2\pi\,0.7\,t)$ and $0.5\cos(2\pi\,2.3\,t)$, separately and together. Then it moves the ground at random: one Gaussian number of rms 1 every 5 ms, for 20 s. It finds the motion of the mirror twice: by integrating the equation and by the product $G\,X_{\mathrm{in}}$, computed with `np.fft` in `apply_transfer`.

**Needs.** `rhs` from the free oscillator; `transfer` from task 1.

In [ ]:
def respond(m, k, b, x_in, t):
    """x_out on the times t, from rest, for the ground motion x_in(t)."""
    sol = solve_ivp(rhs, (0, t[-1]), [0.0, 0.0], t_eval=t, args=(m, k, b, x_in),
                    rtol=1e-10, atol=1e-12, max_step=t[1] - t[0])
    x_out = sol.y[0]
    return x_out


def apply_transfer(m, k, b, x_in, fs):
    """x_out as the product G X_in in frequency; zeros added for the ringing to die."""
    n = 8 * x_in.size
    f = np.fft.rfftfreq(n, d=1 / fs)
    x_out = np.fft.irfft(transfer(f, m, k, b) * np.fft.rfft(x_in, n=n), n=n)[:x_in.size]
    return x_out

In [ ]:
# change these: mass m (kg), spring constant k (N/m), damping b (kg/s)
m, k, b = 1.0, 39.5, 0.21
FS = 200.0
t = np.arange(0, 20, 1 / FS)

def x_in_a(s):
    x_in = np.cos(2 * np.pi * 0.7 * s)
    return x_in


def x_in_b(s):
    x_in = 0.5 * np.cos(2 * np.pi * 2.3 * s)
    return x_in


def x_in_ab(s):
    x_in = x_in_a(s) + x_in_b(s)
    return x_in


x_out_a, x_out_b, x_out_ab = (respond(m, k, b, x, t) for x in (x_in_a, x_in_b, x_in_ab))
print(f"two sinusoids: largest |x_out,ab - x_out,a - x_out,b| = "
      f"{np.max(np.abs(x_out_ab - x_out_a - x_out_b)):.1e} on a signal of {np.max(np.abs(x_out_ab)):.2f}")

x_in_noise = np.random.default_rng(20150914).normal(0.0, 1.0, t.size)


def x_in_random(s):
    x_in = np.interp(s, t, x_in_noise)
    return x_in


x_out_ivp = respond(m, k, b, x_in_random, t)
x_out_product = apply_transfer(m, k, b, x_in_noise, FS)
print(f"random ground: rms of x_out = {x_out_ivp.std():.3f}, largest difference of the"
      f" product / rms = {np.max(np.abs(x_out_product - x_out_ivp)) / x_out_ivp.std():.3f}")

In [ ]:
fig, (axa, axb) = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
axa.plot(t, x_out_ab, color=P['BLUE'], lw=2.0, label='both sinusoids at once')
axa.plot(t, x_out_a + x_out_b, color=P['BRICK'], lw=1.0, ls='--', label='the two outputs added')
axa.set(ylabel=r'$x_{\mathrm{out}}$')
axa.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
axb.plot(t, x_in_noise, color=P['GREY_MID'], lw=0.5, label=r'$x_{\mathrm{in}}$, random')
axb.plot(t, x_out_ivp, color=P['BLUE'], lw=1.6, label=r'$x_{\mathrm{out}}$, integration')
axb.plot(t, x_out_product, color=P['BRICK'], lw=1.0, ls='--', label=r'$x_{\mathrm{out}}$, product')
axb.set(xlabel='time (s)', ylabel='motion', ylim=(-4, 4))
axb.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=3, frameon=False)
plt.show()

**What to look for.** The answer to the sum is the sum of the answers. For the random ground the product gives the integration with no loop in time. The mirror does not follow the fast jitter of the ground: it rings near $f_{0}$ and smooths the rest.

**Question.** Which term in the equation of the mirror would break the superposition and what would the output then contain? Why does the random ground make the mirror swing mostly near $f_{0}$?

<details>
<summary>Solution</summary>

The two sinusoids add to about $10^{-9}$, the tolerance of the solver. For the random ground the product agrees with the integration to about $0.02$ of the rms. The integration joins the samples with straight lines and the product treats them as steps. The two pictures of the same ground differ inside each step.

A term that is not linear in $x_{\mathrm{out}}$ breaks it: a spring that stiffens, with a force $-k x_{\mathrm{out}} - \kappa x_{\mathrm{out}}^{3}$. The output for the sum would contain frequencies that neither input carries, such as $3\times0.7$ Hz and $2\times0.7 + 2.3$ Hz and the difference would grow with the amplitude. The random ground moves at every frequency with the same strength. The mirror multiplies each frequency by $|G|$: $Q$ times near $f_{0}$, 1 below, $(f_{0}/f)^{2}$ above. So the output is a ringing near $f_{0}$ with a slowly changing amplitude.

</details>


## Part C · The resonance

Task 1 showed a peak in $|G|$ and a phase that crosses $-90^{\circ}$. Task 3 asks what a resonance is and why it sits where it does. Task 4 asks what sets its height and its width.

### Task 3 · What a resonance is: amplitude, phase and power

A resonance is the frequency where the drive gives the mirror the most. Three things mark it. At the natural frequency $f_{0}$ the phase is $-90^{\circ}$ and $|G| = Q$. The drive gives the most power there. The peak of $|G|$ lies just below, at

$$
f_{r} = f_{0}\sqrt{1 - \frac{1}{2Q^{2}}} .
$$

The cell sweeps $f$ from $f_{0}/10$ to $10f_{0}$ with `steady_response` of task 1 and draws the amplitude, the phase and the mean power of the integration as dots on the formulas. The power for a ground amplitude of 1 is $\bar{P} = -\tfrac{1}{2}\,k\,\omega\,\mathrm{Im}\,G$. The cell prints where the amplitude, the power and the phase of $-90^{\circ}$ fall. One sweep takes about a minute at $Q = 30$.

**Needs.** `oscillator_parameters` from the free oscillator; `transfer` and `steady_response` from task 1.

> **The symbols**
>
> - **$f_{r}$** — the frequency of the peak of $|G|$
> - **$\mathrm{Im}\,G$** — the imaginary part of $G$

In [ ]:
# change these: mass m (kg), spring constant k (N/m), damping b (kg/s)
m, k, b = 1.0, 39.5, 0.21

f0, Q, tau0 = oscillator_parameters(m, k, b)
f_sweep = f0 * np.unique(np.round(np.concatenate([np.geomspace(0.1, 10, 15),
                                                  np.linspace(0.8, 1.2, 21)]), 6))
amplitude, phase, power = np.array([steady_response(m, k, b, fi) for fi in f_sweep]).T

f_fine = f0 * np.geomspace(0.1, 10, 400001)
G = transfer(f_fine, m, k, b)
power_formula = -0.5 * k * 2 * np.pi * f_fine * G.imag       # ground amplitude 1
print(f"Q = {Q:.2f}   f0 = {f0:.5f} Hz   f_r = {f0 * np.sqrt(1 - 1 / (2 * Q ** 2)):.5f} Hz")
print(f"largest amplitude {np.abs(G).max():.3f} at {f_fine[np.argmax(np.abs(G))]:.5f} Hz")
print(f"largest power at {f_fine[np.argmax(power_formula)]:.5f} Hz,"
      f" phase -90 deg at {f_fine[np.argmin(np.abs(np.angle(G, deg=True) + 90))]:.5f} Hz")
mismatch = np.max(np.abs(amplitude / np.abs(transfer(f_sweep, m, k, b)) - 1))
print(f"integration against |G|: largest relative difference {mismatch:.1e}")

In [ ]:
fig, (axa, axb, axc) = plt.subplots(3, 1, figsize=(9, 8.5), sharex=True)
axa.loglog(f_fine, np.abs(G), color=P['BLUE'], lw=1.5, label='$|G|$, formula')
axa.loglog(f_sweep, amplitude, 'o', color=P['INK'], ms=3.5, label='integration')
axa.set(ylabel=r'amplitude of $x_{\mathrm{out}}$')
axa.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
axb.semilogx(f_fine, np.angle(G, deg=True), color=P['BLUE'], lw=1.5)
axb.semilogx(f_sweep, phase, 'o', color=P['INK'], ms=3.5)
axb.set(ylabel=r'phase $\phi$ (degrees)', yticks=[-180, -135, -90, -45, 0])
axc.loglog(f_fine, power_formula, color=P['BRICK'], lw=1.5)
axc.loglog(f_sweep, power, 'o', color=P['INK'], ms=3.5)
axc.set(xlabel='frequency of the ground motion (Hz)', ylabel=r'mean power $\bar{P}$ (W)')
for ax in (axa, axb, axc):
    ax.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
plt.show()

**What to look for.** The dots of the integration lie on the formula for the amplitude, the phase and the power. The power has one peak and it is narrow. The amplitude peaks a little lower in frequency.

**Question.** Raise $b$ to $2.1$ and then to $6$ kg/s. Where are the largest amplitude, the largest power and the phase of $-90^{\circ}$? Why does the drive give the most power exactly at $f_{0}$?

<details>
<summary>Solution</summary>

At $Q = 29.9$ the largest amplitude is $29.93$ at $f_{r} = 0.99999$ Hz. The largest power and the phase of $-90°$ are both at $f_{0} = 1.00027$ Hz, three parts in $10^{4}$ higher. The integration agrees with the formulas to the tolerance of the solver. At $b = 2.1$ kg/s ($Q = 3.0$) the peak is lower, $3.04$, much wider and at $f_{r} = 0.972f_{0}$; the power and the phase still mark $f_{0}$. At $b = 6$ kg/s ($Q = 1.05$) the peak is $1.19$ high at $0.74f_{0}$ and is barely a peak. Below $Q = 1/\sqrt{2}$ there is none.

At $f_{0}$ the phase is $-90°$: when the ground moves as a cosine, the mirror moves as a sine and its velocity as a cosine. The velocity is **in phase with the force** of the spring. The drive pushes the mirror the way it already moves in every part of every cycle, so it gives the most power. At $f_{0}$ the spring and the inertia cancel, $k = m\omega_{0}^{2}$ and $G(\omega_{0}) = k/(\mathrm{i} b\omega_{0}) = -\mathrm{i} Q$: only the friction limits the motion. The amplitude peaks a little lower, because at a lower frequency the same velocity is a larger displacement. For a high $Q$ the three frequencies merge into one number, the resonance.

</details>


### Task 4 · The height and the width of the resonance: the same $Q$ in frequency and in time

The resonance has a height and a width. Lecture 8 says the height is $Q$ and the full width at half power, where $|G|$ is its peak divided by $\sqrt{2}$, is $\Delta\omega \simeq \omega_{0}/Q$ for $Q \gg 1$. The free oscillator gave the decay time $\tau_{0} = 2Q/\omega_{0}$. Their product is

$$
\tau_{0}\,\Delta\omega = \frac{2Q}{\omega_{0}}\,\frac{\omega_{0}}{Q} = 2 .
$$

The function `peak_numbers` reads the height, the frequency and the width of the peak of $|G|$. The cell draws $|G|$ around $f_{0}$ for three values of $b$ and prints, for each, the height next to $Q$, the width next to $f_{0}/Q$ and $\tau_{0}\Delta\omega$.

**Needs.** `oscillator_parameters` from the free oscillator; `transfer` from task 1.

> **The symbols**
>
> - **$\Delta f$** — the full width at half power of the peak, in hertz; $\Delta\omega = 2\pi\Delta f$

In [ ]:
def peak_numbers(f, gain):
    """Height, frequency and full width at half power of the peak of gain."""
    i = np.argmax(gain)
    inside = f[gain >= gain[i] / np.sqrt(2)]
    numbers = (gain[i], f[i], inside[-1] - inside[0])
    return numbers

In [ ]:
# change these: mass m (kg), spring constant k (N/m), three values of the damping b (kg/s)
m, k, b_values = 1.0, 39.5, (2.1, 0.63, 0.21)

f = np.linspace(0.01, 3.0, 600001)
fig, ax = plt.subplots(figsize=(9, 4))
for b, colour in zip(b_values, (P['BLUE'], P['SAGE'], P['BRICK'])):
    f0, Q, tau0 = oscillator_parameters(m, k, b)
    height, f_peak, width = peak_numbers(f, np.abs(transfer(f, m, k, b)))
    print(f"b = {b:5.2f} kg/s: Q = {Q:6.2f}   height {height:6.2f}   width {width:.4f} Hz"
          f" (f0/Q = {f0 / Q:.4f} Hz)   tau0 dw = {tau0 * 2 * np.pi * width:.3f}")
    ax.plot(f, np.abs(transfer(f, m, k, b)), color=colour, lw=1.5, label=f'Q = {Q:.0f}')
ax.set(xlabel='frequency (Hz)', ylabel='$|G|$', xlim=(0.5, 1.5), ylim=(0.5, 50), yscale='log')
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=3, frameon=False)
plt.show()

**What to look for.** The higher $Q$, the taller and the narrower the peak. The width is close to $f_{0}/Q$ and $\tau_{0}\Delta\omega$ is close to 2.

**Question.** Change $m$ and then $k$. Which parameter moves the peak, which changes its height and which its width? Why is $\tau_{0}\Delta\omega = 2$ for every $b$ and why only roughly at $Q = 3$?

<details>
<summary>Solution</summary>

For $Q = 3.0$, $10.0$ and $29.9$ the height is $3.04$, $9.99$ and $29.93$, a little above $Q$. The width is $0.345$, $0.1005$ and $0.0334$ Hz against $f_{0}/Q = 0.334$, $0.1003$ and $0.0334$ Hz and $\tau_{0}\Delta\omega$ is $2.06$, $2.005$ and $2.000$.

The peak sits at $f_{0} = \sqrt{k/m}/2\pi$: a stiffer spring moves it up, a heavier mass down. The width is $\Delta\omega = \omega_{0}/Q = b/m$: friction over mass, nothing else. The height $Q = \sqrt{km}/b$ grows with both. The decay time $\tau_{0} = 2m/b$ is set by the same ratio, so $\tau_{0}\Delta\omega = 2$ for every $b$: a narrow peak and a long ringing are one fact, seen in frequency and in time. The width $\omega_{0}/Q$ holds for a narrow, symmetric peak, $Q \gg 1$. At $Q = 3$ the peak is a third of $f_{0}$ wide and the result is rough.

</details>


## Part D · At the resonance and away from it

A setup is built by choosing $m$, $k$ and $b$ and so $f_{0}$ and $Q$, for the job. Below $f_{0}$ the mirror follows the input. At $f_{0}$ it amplifies the input by $Q$, in a band $f_{0}/Q$ wide. Far above $f_{0}$ it isolates: $|G| \simeq (f_{0}/f)^{2}$. Tasks 5 and 6 work at the resonance: a resonant bar and a violin mode. Task 7 works away from it: the mass follows far below and isolates far above.

### Task 5 · A resonant bar: working at the resonance

The first gravitational-wave detectors were resonant bars: metal cylinders of a few tonnes that ring when a wave passes (Allen and others 2000). A bar must answer a weak force as strongly as it can. So it works at its resonance, near a kilohertz, with a high $Q$. The cell takes a mass of 1000 kg, a spring constant that puts $f_{0}$ near 900 Hz and a damping that gives $Q$ near $10^{6}$. They are round numbers for this task and not the values of one instrument.

The cell prints $f_{0}$, $Q$, the height and the width of the peak, $|G|$ away from the peak, the decay time in seconds and in swings and $\tau_{0}\Delta\omega$. It draws the peak and the envelope of the ringing.

**Needs.** `oscillator_parameters` from the free oscillator; `transfer` from task 1; `peak_numbers` from task 4.

In [ ]:
# change these: mass m (kg), spring constant k (N/m), damping b (kg/s)
m, k, b = 1000.0, 3.2e10, 5.7

f0, Q, tau0 = oscillator_parameters(m, k, b)
offset = np.linspace(-0.01, 0.01, 200001)                 # Hz around f0
gain = np.abs(transfer(f0 + offset, m, k, b))
height, f_peak, width = peak_numbers(offset, gain)
print(f"f0 = {f0:.2f} Hz   Q = {Q:.3e}   height of the peak {height:.3e}")
print(f"width = {width * 1000:.3f} mHz   f0/Q = {f0 / Q * 1000:.3f} mHz")
for far in (0.01, 1.0, 100.0):
    print(f"{far:7.2f} Hz above f0: |G| / height = {abs(transfer(f0 + far, m, k, b)) / height:.1e}")
print(f"tau0 = {tau0:.0f} s = {tau0 * f0:.2e} swings   tau0 dw = {tau0 * 2 * np.pi * width:.3f}")

fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 3.8))
axa.plot(offset * 1000, gain / height, color=P['BLUE'], lw=1.5)
axa.axhline(1 / np.sqrt(2), color=P['GREY_MID'], lw=0.8, ls=':')
axa.set(xlabel='frequency from $f_0$ (mHz)', ylabel='$|G|$ / height', xlim=(-3, 3))
t_ring = np.linspace(0, 4 * tau0, 400)
axb.plot(t_ring / 60, np.exp(-t_ring / tau0), color=P['BRICK'], lw=1.5)
axb.axvline(tau0 / 60, color=P['GREY_MID'], lw=0.8, ls=':')
axb.set(xlabel='time (minutes)', ylabel='envelope of the ringing')
plt.show()

**What to look for.** The response at the peak is $Q$ times the response at low frequency, but only in a band $f_{0}/Q$ wide, less than one millihertz. One hertz away it is thousands of times smaller. So a bar hears one narrow band. The price is time: it rings for minutes after a kick.

**Question.** Make $b$ ten times larger, then ten times smaller. What happens to the gain, to the band the bar hears and to the time it rings? Can a bar have a wide band and a short ringing with the same gain?

<details>
<summary>Solution</summary>

With $m = 1000$ kg, $k = 3.2\times10^{10}$ N/m and $b = 5.7$ kg/s, $f_{0} = 900.3$ Hz and $Q = 9.9\times10^{5}$. The width is $0.91$ mHz, which is $f_{0}/Q$ and one hertz above the peak $|G|$ is $4.5\times10^{-4}$ of the height. The bar rings for $\tau_{0} = 351$ s, about $3\times10^{5}$ swings and $\tau_{0}\Delta\omega = 2.00$.

Ten times more friction makes the gain ten times smaller, the band ten times wider and the ringing ten times shorter; ten times less friction does the opposite. The gain $Q$, the band $f_{0}/Q$ and the ringing $2Q/\omega_{0}$ are tied together. A high gain comes only with a narrow band and a long ringing. That is what working at the resonance means: amplification, in a narrow band, paid for in time. A bar hears a signal that lies in its band and lasts long enough to fill it.

</details>


### Task 6 · A violin mode: a high $Q$ keeps the noise in a narrow line

A mirror hangs from fibres. A fibre rings like a string and its violin modes lie at a few hundred hertz, a fundamental near 430 Hz in Advanced Virgo and harmonics above it. In the data they are narrow lines. Each mode is an oscillator, pushed at random by the heat of the fibre. That push has the same strength at every frequency, with a level proportional to the loss $1/Q$. We quote this (Saulson 2017). The mirror then moves at each frequency with the strength of the push times $|G|^{2}$:

$$
S(f) \propto \frac{1}{Q}\,\left|G(f)\right|^{2} .
$$

The function `thermal_line` computes $S$ for one mode. The cell draws it for two values of $b$, a low $Q$ and a high $Q$ and prints the height of each line, its width and its area: the total motion of the mode. The numbers are round, for this task.

**Needs.** `oscillator_parameters` from the free oscillator; `transfer` from task 1; `peak_numbers` from task 4.

> **The symbols**
>
> - **$S(f)$** — the strength of the motion of the mode at each frequency, per hertz, in arbitrary units

In [ ]:
def thermal_line(f, m, k, b):
    """The motion per hertz of a mode pushed by a thermal force of level 1/Q."""
    f0, q, tau0 = oscillator_parameters(m, k, b)
    s = np.abs(transfer(f, m, k, b)) ** 2 / q
    return s

In [ ]:
# change these: mass m (kg), spring constant k (N/m), two values of the damping b (kg/s)
m, k, b_values = 1.0, 9.87e6, (157.0, 1.57)

f = np.linspace(1.0, 1000.0, 2000001)
fig, ax = plt.subplots(figsize=(9, 4))
for b, colour in zip(b_values, (P['BLUE'], P['BRICK'])):
    f0, Q, tau0 = oscillator_parameters(m, k, b)
    S = thermal_line(f, m, k, b)
    height, f_peak, width = peak_numbers(f, np.sqrt(S))
    area = np.trapezoid(S, f)
    print(f"Q = {Q:6.0f}: height {height ** 2:8.2f}   width {width:6.3f} Hz (f0/Q = {f0 / Q:.3f} Hz)"
          f"   area {area:.2f}   at 300 Hz {thermal_line(300.0, m, k, b):.2e}")
    ax.semilogy(f, S, color=colour, lw=1.2, label=f'Q = {Q:.0f}')
ax.set(xlabel='frequency (Hz)', ylabel='$S(f)$ (arbitrary units)', xlim=(400, 600))
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
plt.show()

**What to look for.** The line of the high-$Q$ mode is a hundred times higher and a hundred times narrower than the line of the low-$Q$ mode. The area is the same. Far from the line the high-$Q$ mode is a hundred times quieter.

**Question.** Change the second $b$ from $1.57$ to $15.7$ kg/s. What happens to the height, the width and the area of the line? Why does the area not change and why do we want fibres with a high $Q$?

<details>
<summary>Solution</summary>

For $Q = 20$ the line is 20 high and 25 Hz wide; for $Q = 2001$ it is 2001 high and $0.25$ Hz wide. The area is $784$ and $785$, the same to the step of the frequency grid: $\pi f_{0}/2$ in these units. At 300 Hz the high-$Q$ mode is a hundred times lower. With $b = 15.7$ kg/s ($Q = 200$) the line is ten times lower and ten times wider than at $Q = 2000$ and the area is again the same.

The push grows with the loss, $1/Q$. The peak of $|G|^{2}$ grows as $Q^{2}$ over a width $f_{0}/Q$, so its area grows as $Q$. The two cancel: the total motion of the mode does not depend on $Q$. It is the energy of the mode in thermal equilibrium, set by the temperature. A high $Q$ squeezes that motion into a narrow line and leaves the rest of the band quieter. So fibres with a high $Q$ keep their thermal noise in lines a fraction of a hertz wide, which the analysis can remove and the detector is quiet between them.

</details>


### Task 7 · Away from the resonance: the mass follows below and isolates above

Away from the resonance the mirror does one of two things. Far below $f_{0}$ it follows the ground: $|G| \simeq 1$ and $\phi \simeq 0^{\circ}$. Far above $f_{0}$ it stays behind: $|G| \simeq (f_{0}/f)^{2}$ and $\phi \simeq -180^{\circ}$. A ground-based detector measures between 10 and 100 Hz. We want that band far above the resonance of the mirror.

The function `draw_regimes` draws $|G|$ and its phase for the chosen $m$, $k$ and $b$. It marks $f_{0}$, the band $f_{0} \pm f_{0}/2Q$ of the resonance and the line $(f_{0}/f)^{2}$ and shades the band from 10 to 100 Hz. The cell prints $f_{0}$, $Q$ and then $|G|$ and the phase far below the resonance, at $f_{0}/100$ and at 10 and 100 Hz.

**Needs.** `oscillator_parameters` from the free oscillator; `transfer` from task 1.

In [ ]:
def draw_regimes(m, k, b):
    """|G| and its phase, with f0, the resonance band and (f0/f)^2 marked."""
    f0, q, tau0 = oscillator_parameters(m, k, b)
    f = np.geomspace(min(0.01, f0 / 30), max(1000.0, 30 * f0), 20000)
    G = transfer(f, m, k, b)
    fig, (axa, axb) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    axa.loglog(f, np.abs(G), color=P['BLUE'], lw=1.6, label='$|G|$')
    above = f > 3 * f0
    axa.loglog(f[above], (f0 / f[above]) ** 2, color=P['BRICK'], lw=1.2, ls='--',
               label='$(f_0/f)^2$')
    for ax in (axa, axb):
        ax.axvspan(10, 100, color=P['SAGE'], alpha=0.15)
        ax.axvspan(f0 * (1 - 1 / (2 * q)), f0 * (1 + 1 / (2 * q)), color=P['AMBER'], alpha=0.3)
        ax.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
    axa.set(ylabel='$|G|$')
    axa.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
    axb.semilogx(f, np.angle(G, deg=True), color=P['BLUE'], lw=1.6)
    axb.set(xlabel='frequency (Hz)', ylabel='phase (degrees)', yticks=[-180, -90, 0])
    plt.show()
    params = (f0, q)
    return params

In [ ]:
# change these: mass m (kg), spring constant k (N/m), damping b (kg/s)
m, k, b = 40.0, 570.0, 0.15

f0, Q = draw_regimes(m, k, b)
print(f"f0 = {f0:.3f} Hz   Q = {Q:.0f}   band of the resonance {f0 / Q * 1000:.2f} mHz")
for f_i in (f0 / 100, 10.0, 100.0):
    g_i = transfer(f_i, m, k, b)
    print(f"f = {f_i:8.3f} Hz: |G| = {abs(g_i):.2e}   phase = {np.angle(g_i, deg=True):7.2f} deg")
for f_i in (10.0, 100.0):
    print(f"(f0/f)^2 at {f_i:5.0f} Hz = {(f0 / f_i) ** 2:.2e}")

**What to look for.** The green band is where the detector measures. The amber band marks the resonance, $f_{0}/Q$ wide: at $Q = 1000$ it is a thin line. Here $f_{0}$ is near 0.6 Hz. Far below it, at $f_{0}/100$, the mirror follows the ground. The detector band lies in the third regime, where $|G|$ is small and falls as $(f_{0}/f)^{2}$.

**Question.** We want the ground motion between 10 and 100 Hz to reach the mirror as little as possible. Change $k$, then $m$, then $b$. Which $f_{0}$ and which $Q$ do we want and why does $Q$ hardly matter at 10 Hz? What does the mirror do with a ground motion at $f_{0}/100$?

<details>
<summary>Solution</summary>

With $m = 40$ kg, $k = 570$ N/m and $b = 0.15$ kg/s, $f_{0} = 0.601$ Hz and $Q = 1007$. At 10 Hz $|G| = 3.6\times10^{-3}$ and at 100 Hz $3.6\times10^{-5}$, which is $(f_{0}/f)^{2}$.

Far above the resonance $|G| \simeq (f_{0}/f)^{2} = k/(m\omega^{2})$: the inertia of the mass wins over the spring and the mass stays still. So we want $f_{0}$ as low as possible, a soft spring and a heavy mass: ten times lower $f_{0}$ means a hundred times better isolation. The damping enters $G$ through $\mathrm{i} b\omega$, which far above $f_{0}$ is small next to $m\omega^{2}$: $Q$ changes only the peak and its width, both far below the band. A high $Q$ puts a tall peak at 0.6 Hz, which is outside the band and is damped by other means.

At $f_{0}/100$, about 6 mHz, $|G| = 1.000$ and the phase is $-0.0$ degrees: the spring wins over the inertia and over the friction. The mirror moves with the ground. This is the first regime. A mirror isolates only above its resonance. Below it the suspension does nothing.

</details>


## Part E · Several pendulums

One stage is not enough. The next task asks how many stages and how long they must be.

### Task 8 · A chain of pendulums: how many and how long

A mirror hangs from a pendulum. A pendulum of length $\ell$ is an oscillator with the spring constant $k = m g_{\oplus}/\ell$, so its natural frequency is $f_{p} = \sqrt{g_{\oplus}/\ell}\,/2\pi$. Its fibres lose energy: the **loss angle** $\phi_{\mathrm{loss}}$ is the fraction of the energy lost per radian of swing, so it plays the role of $1/Q$ (Saulson 2017). The transfer function of systems in a chain is the product of their transfer functions, so $N$ identical stages give $|G|^{N}$. Far above $f_{p}$ that is $(f_{p}/f)^{2N}$.

The ground moves as $10^{-7}/f^{2}\ \mathrm{m}/\sqrt{\mathrm{Hz}}$ between 1 and 100 Hz. This is an assumed model of a quiet site for this task, not a measurement. A 4 km interferometer that must reach $h = 10^{-23}/\sqrt{\mathrm{Hz}}$ at 10 Hz needs a mirror motion below $hL_{\mathrm{arm}} = 4\times10^{-20}\ \mathrm{m}/\sqrt{\mathrm{Hz}}$. The function `stage` returns $|G|$ of one pendulum. The cell draws the motion of the mirror after 0 to $N + 1$ stages with the requirement and prints the motion at 10 Hz and the frequency above which $N$ stages meet the requirement.

**Needs.** `transfer` from task 1.

> **The symbols**
>
> - **$\ell$** — the length of one pendulum
> - **$g_{\oplus} = 9.81\ \mathrm{m/s^{2}}$** — the acceleration of gravity
> - **$f_{p}$** — the natural frequency of one pendulum
> - **$\phi_{\mathrm{loss}}$** — the loss angle of the fibre: it plays the role of $1/Q$
> - **$N$** — the number of stages
> - **$L_{\mathrm{arm}}$** — the length of an arm of the interferometer, 4 km

In [ ]:
g_earth = 9.81                               # g_oplus, in m/s^2


def stage(f, m, ell, phi_loss):
    """|G| of one pendulum of mass m and length ell, with Q = 1 / phi_loss."""
    k = m * g_earth / ell
    b = np.sqrt(k * m) * phi_loss
    gain = np.abs(transfer(f, m, k, b))
    return gain

In [ ]:
# change these: number of stages N, mass m (kg), length ell (m) and loss angle of each stage
N, m, ell, phi_loss = 4, 40.0, 0.69, 1e-3

f = np.geomspace(1, 100, 2000)
ground = 1e-7 / f ** 2                       # m / sqrt(Hz)
required = 1e-23 * 4000                      # strain 1e-23 over 4 km
print(f"f_p = {np.sqrt(g_earth / ell) / (2 * np.pi):.3f} Hz;"
      f" one stage divides the motion at 10 Hz by {1 / stage(10.0, m, ell, phi_loss):.0f}")
fig, ax = plt.subplots(figsize=(9, 5))
for n in range(N + 2):
    at10 = 1e-7 / 10.0 ** 2 * stage(10.0, m, ell, phi_loss) ** n
    print(f"{n} stages: {at10:.2e} m/rtHz at 10 Hz, {at10 / required:9.3g} times the requirement")
    ax.loglog(f, ground * stage(f, m, ell, phi_loss) ** n, lw=1.4, label=f'{n} stage' + ('s' if n != 1 else ''))
ax.axhline(required, color=P['INK'], lw=1.2, ls='--', label='requirement')
ax.set(xlabel='frequency (Hz)', ylabel=r'mirror motion (m/$\sqrt{\mathrm{Hz}}$)', ylim=(1e-24, 1e-6))
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=4, frameon=False, fontsize=9)
plt.show()
meets = f[ground * stage(f, m, ell, phi_loss) ** N < required]
print(f"{N} stages meet the requirement above {meets[0]:.1f} Hz" if meets.size else "never met")

**What to look for.** Each stage divides the motion by $(f/f_{p})^{2}$ above its resonance, so the curves get steeper with $N$: the slope is $-(2 + 2N)$ with the ground. Four stages are still a little short at 10 Hz. Five are enough.

**Question.** Change the mass, then the length, then $N$. Why does the mass not change the isolation? How many stages of 0.69 m meet the requirement at 10 Hz? How long must four stages be?

<details>
<summary>Solution</summary>

With $\ell = 0.69$ m, $f_{p} = 0.600$ Hz and one stage divides the motion at 10 Hz by 277. The mirror motion at 10 Hz is $1.0\times10^{-9}$, $3.6\times10^{-12}$, $1.3\times10^{-14}$, $4.7\times10^{-17}$, $1.7\times10^{-19}$ and $6.2\times10^{-22}\ \mathrm{m}/\sqrt{\mathrm{Hz}}$ for 0 to 5 stages. Four stages are a factor $4.27$ short and meet the requirement only above $11.6$ Hz; five stages are enough.

The spring constant of a pendulum, $k = m g_{\oplus}/\ell$, grows with the mass as the inertia does, so $f_{p}^{2} = g_{\oplus}/(4\pi^{2}\ell)$ does not depend on $m$. Only the length sets the isolation of one stage, $(f_{p}/f)^{2} \propto 1/\ell$. Four stages gain $\ell^{4}$, so a factor $4.3$ needs $\ell$ larger by $4.3^{1/4} = 1.44$: about 1 m per stage. The exponent of the slope is a count of pendulums. Open `H04`, find the seismic term of the noise budget and account for its exponent 10.

</details>


## What to hand in

The answer to the question of each task, in two or three sentences, with the parameters you changed and what you saw:

1. **A · The transfer function.** The values of $|G|$ and $\phi$ below, at and above $f_{0}$. Which two numbers fix the curve.
2. **B · Any input.** Why one product gives the output for any input. What would break the superposition.
3. **C · The resonance.** Where the largest amplitude, the largest power and the phase of $-90^{\circ}$ lie. What sets the height and the width of the peak and why $\tau_{0}\Delta\omega = 2$.
4. **D · At the resonance and away from it.** What a bar gains at its resonance and what it pays. Why a high $Q$ keeps a violin mode in a narrow line. Which $f_{0}$ and $Q$ isolate a mirror and what the mirror does far below $f_{0}$.
5. **E · Several pendulums.** How many stages meet the requirement and how long four stages must be. Why only the length of a pendulum matters.

> **Where this goes next**
>
> You have found the transfer function of the oscillator and used it for any input. You have seen what a resonance is and what it means to work at it and away from it. Below the natural frequency the mass follows the input. At the resonance it amplifies, in a narrow band. Far above it isolates. Every stage of a chain isolates again. A detector uses all three and all of it rests on linearity.

## Software

`numpy` throughout. `solve_ivp` integrates the equation of the oscillator and `np.fft` gives the product $G\,X_{\mathrm{in}}$. The oscillator, the loss angle and the pendulum isolation are treated as in (Saulson 2017). The transforms follow (Oppenheim et al. 1997). The resonant bars are described in (Allen and others 2000).

**References**

- Saulson (2017). *Fundamentals of Interferometric Gravitational Wave Detectors*.
- Oppenheim et al. (1997). *Signals and Systems*.
- Allen and others (2000). *First Search for Gravitational Wave Bursts with a Network of Detectors*.